# Fashion RAG Chatbot — ViFashionCLIP Vietnamese Embedding

Notebook này được tạo từ `Chatbot_RAG_MultiModal.ipynb`, thay embedding kho sản phẩm Layer A bằng **FashionCLIP tiếng Việt đã fine-tune**.

- **Layer A (sản phẩm):** `AITeamVN/Vietnamese_Embedding_v2 + ProjectionHead Stage 2` → vector **512 chiều** trong không gian FashionCLIP.
- **Layer B (quy tắc phối đồ):** giữ `BGE-M3` qua Ollama → vector **1024 chiều** cho rule text thuần.
- **Image encoder:** nếu cần search bằng ảnh, dùng FashionCLIP image encoder gốc.


## PHẦN 1: Import thư viện

In [2]:
import json, sys, uuid, os, time, base64, re
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.models import PointStruct
from qdrant_client.http.models import Filter, FieldCondition, MatchAny
from langchain_core.documents import Document
from tqdm import tqdm
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.retrievers import BaseRetriever
from typing import Any
import ollama
from ollama import Client
from PIL import Image
import io
print("[OK] Import hoàn tất!")


[OK] Import hoàn tất!


## PHẦN 2: Embedding Models — ViFashionCLIP cho sản phẩm, BGE-M3 cho Layer B

- **BGEM3Embeddings**: Wrapper cho BGE-M3 qua Ollama, dùng cho Layer B (quy tắc phối đồ). Vector 1024 chiều.
- **ViFashionCLIPTextEmbeddings**: Wrapper cho ViFashionCLIP tiếng Việt đã fine-tune, dùng cho Layer A (sản phẩm). Vector 512 chiều.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor

# ════════════════════════════════════════════════════════════════
# CONSTANTS — Tập trung tên model để đổi 1 chỗ duy nhất
# ════════════════════════════════════════════════════════════════
LLM_MODEL    = "qwen3:4b-instruct"   # LLM chính cho chat + intent + summarize
QWEN_VL_MODEL = "qwen2.5vl:3b"       # Vision-Language cho phân tích ảnh

# ════════════════════════════════════════════════════════════════
# 1) BGE-M3: Embedding cho Layer B (quy tắc phối đồ / text thuần)
#    Vector: 1024 chiều
# ════════════════════════════════════════════════════════════════
class BGEM3Embeddings(Embeddings):
    """BGE-M3 wrapper qua Ollama. Dùng cho Layer B: phong cách, bối cảnh,
    dáng người, tone da, và lý do tư vấn. Vector 1024 chiều."""

    def __init__(self, model_name="bge-m3"):
        self.ollama_embeddings = OllamaEmbeddings(
            model=model_name,
            base_url="http://localhost:11434"
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.ollama_embeddings.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        return self.ollama_embeddings.embed_query(text)


# ════════════════════════════════════════════════════════════════
# 2) ViFashionCLIP: Embedding cho Layer A (sản phẩm tiếng Việt)
#    Vector: 512 chiều
# ════════════════════════════════════════════════════════════════
TEACHER_MODEL_NAME = "patrickjohncyh/fashion-clip"
STUDENT_MODEL_NAME = "AITeamVN/Vietnamese_Embedding_v2"
STUDENT_MAX_LENGTH = 128
PROJECTION_HIDDEN_DIM = 1024
PROJECTION_NUM_LAYERS = 3
PROJECTION_DROPOUT = 0.05


def resolve_chatbot_fashion_dir() -> Path:
    """Tìm thư mục gốc Chatbot_Fashion từ vị trí hiện tại."""
    cwd = Path.cwd()
    if cwd.name == "notebooks" and cwd.parent.name == "Chatbot_Fashion":
        return cwd.parent
    if cwd.name == "Chatbot_Fashion":
        return cwd
    candidate = cwd / "Chatbot_Fashion"
    if candidate.exists():
        return candidate
    for parent in cwd.parents:
        candidate = parent / "Chatbot_Fashion"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục Chatbot_Fashion từ thư mục hiện tại")


CHATBOT_FASHION_DIR = resolve_chatbot_fashion_dir()
VIFASHIONCLIP_CHECKPOINT = (
    CHATBOT_FASHION_DIR
    / "Vietnamese"
    / "vifashionclip_aiteamvn_embedding_v2_projection_336k"
    / "stage2_last_layers"
    / "best_stage2_model.pt"
)


class ResidualMLPBlock(nn.Module):
    def __init__(self, dim, dropout=0.05):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)


class ProjectionHead(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=1024, num_layers=3, dropout=0.05):
        super().__init__()
        layers = [
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        ]
        for _ in range(max(0, num_layers - 2)):
            layers.append(ResidualMLPBlock(hidden_dim, dropout=dropout))
        layers.extend([
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim),
        ])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    denom = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / denom


class Stage2StudentProjection(nn.Module):
    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(outputs.last_hidden_state, attention_mask)
        return self.projection_head(pooled)


class ViFashionCLIPTextEmbeddings(Embeddings):
    """
    LangChain Embeddings wrapper cho ViFashionCLIP tiếng Việt đã fine-tune.

    Vector: 512 chiều.
    Dùng cho:
    - Layer A: embedding sản phẩm tiếng Việt
    - Search query tiếng Việt

    KHÔNG dùng cho Layer B (Layer B dùng BGE-M3).
    """
    def __init__(self, checkpoint_path: str | Path = VIFASHIONCLIP_CHECKPOINT, batch_size: int = 32):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.checkpoint_path = Path(checkpoint_path)
        if not self.checkpoint_path.exists():
            raise FileNotFoundError(f"Không tìm thấy checkpoint ViFashionCLIP: {self.checkpoint_path}")

        checkpoint = torch.load(self.checkpoint_path, map_location="cpu")
        student_name = checkpoint.get("student_model_name", STUDENT_MODEL_NAME)
        teacher_dim = int(checkpoint.get("teacher_dim", 512))

        self.tokenizer = AutoTokenizer.from_pretrained(student_name)
        encoder = AutoModel.from_pretrained(student_name)
        projection = ProjectionHead(
            input_dim=encoder.config.hidden_size,
            output_dim=teacher_dim,
            hidden_dim=PROJECTION_HIDDEN_DIM,
            num_layers=PROJECTION_NUM_LAYERS,
            dropout=PROJECTION_DROPOUT,
        )

        encoder.load_state_dict(checkpoint["encoder_state_dict"])
        projection.load_state_dict(checkpoint["projection_state_dict"])

        self.model = Stage2StudentProjection(encoder, projection).to(self.device).eval()
        for p in self.model.parameters():
            p.requires_grad = False

        self.embedding_dim = teacher_dim
        print(f"[OK] ViFashionCLIP đã tải: {self.checkpoint_path}")
        print(f"     Device: {self.device} | Embedding dim: {self.embedding_dim}")

    @torch.no_grad()
    def _encode(self, texts: list[str]) -> list[list[float]]:
        all_embeds = []
        for i in tqdm(range(0, len(texts), self.batch_size), desc="ViFashionCLIP embed", leave=False):
            batch = [str(x) for x in texts[i:i + self.batch_size]]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=STUDENT_MAX_LENGTH,
                return_tensors="pt",
            ).to(self.device)
            embeds = self.model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            embeds = F.normalize(embeds, p=2, dim=-1)
            all_embeds.append(embeds.detach().cpu())
        return torch.cat(all_embeds, dim=0).tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._encode(texts)

    def embed_query(self, text: str) -> list[float]:
        return self._encode([text])[0]




class FashionCLIPImageEmbeddings:
    """
    FashionCLIP image encoder for real image retrieval.

    Usage:
    - Product MAIN image -> 512-dim vector
    - User product image -> 512-dim vector

    This class is for image retrieval only. Layer B still uses BGE-M3,
    and product text retrieval still uses ViFashionCLIPTextEmbeddings.
    """
    def __init__(self, model_name: str = TEACHER_MODEL_NAME, batch_size: int = 32):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.model_name = model_name

        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model = CLIPModel.from_pretrained(model_name).to(self.device).eval()
        for p in self.model.parameters():
            p.requires_grad = False

        self.embedding_dim = int(getattr(self.model.config, "projection_dim", 512))
        print(f"[OK] FashionCLIP image encoder loaded: {model_name}")
        print(f"     Device: {self.device} | Embedding dim: {self.embedding_dim}")

    @staticmethod
    def _open_image(image_path: str | Path):
        return Image.open(image_path).convert("RGB")

    @torch.no_grad()
    def encode_image_paths(self, image_paths: list[str | Path]) -> list[list[float] | None]:
        """Encode multiple images. Failed images return None so indexing can skip them."""
        results: list[list[float] | None] = [None] * len(image_paths)

        for start in tqdm(range(0, len(image_paths), self.batch_size), desc="FashionCLIP image embed", leave=False):
            batch_paths = image_paths[start:start + self.batch_size]
            images, valid_positions = [], []

            for offset, path in enumerate(batch_paths):
                try:
                    images.append(self._open_image(path))
                    valid_positions.append(start + offset)
                except Exception as e:
                    print(f"[WARN] Cannot read image {path}: {e}")

            if not images:
                continue

            inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device)
            embeds = self.model.get_image_features(**inputs)
            embeds = F.normalize(embeds, p=2, dim=-1).detach().cpu().tolist()

            for pos, vec in zip(valid_positions, embeds):
                results[pos] = vec

        return results

    def embed_image(self, image_path: str | Path) -> list[float] | None:
        return self.encode_image_paths([image_path])[0]

print("[OK] Các class Embedding đã sẵn sàng")
print(f"     LLM_MODEL   : {LLM_MODEL}")
print(f"     QWEN_VL_MODEL: {QWEN_VL_MODEL}")
print(f"     ViFashionCLIP checkpoint: {VIFASHIONCLIP_CHECKPOINT}")
print(f"     Checkpoint tồn tại: {VIFASHIONCLIP_CHECKPOINT.exists()}")


[OK] Các class Embedding đã sẵn sàng
     LLM_MODEL   : qwen3:4b-instruct
     QWEN_VL_MODEL: qwen2.5vl:3b
     ViFashionCLIP checkpoint: d:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion\Vietnamese\vifashionclip_aiteamvn_embedding_v2_projection_336k\stage2_last_layers\best_stage2_model.pt
     Checkpoint tồn tại: True


## PHẦN 3: Data Pipeline — Indexing sản phẩm vào Qdrant bằng ViFashionCLIP

> Luồng: Đọc file JSONL 65k → chuyển thành LangChain Document → embed bằng ViFashionCLIP → đẩy vào Qdrant.
> Có cơ chế **resume** theo số point hiện có, nên chạy lại sẽ bỏ qua phần đã index.

In [4]:
def extract_image_urls(item: dict) -> list[str]:
    """Lấy danh sách ảnh đại diện từ field images của sản phẩm."""
    return [img.get("large") for img in item.get("images", []) if img.get("large")]


def normalize_to_text(value, default: str = "Không rõ") -> str:
    """Chuyển list/None/value thành chuỗi ngắn gọn để đưa vào page_content."""
    if value is None or value == "":
        return default
    if isinstance(value, list):
        cleaned = [str(x).strip() for x in value if str(x).strip()]
        return ", ".join(cleaned) if cleaned else default
    return str(value).strip() or default


def build_product_metadata(item: dict) -> dict:
    """Metadata dùng để lọc, debug và hiển thị sau khi retrieve."""
    image_urls = extract_image_urls(item)
    return {
        "product_id": item.get("product_id", ""),
        "title":      item.get("title", ""),
        "category":   item.get("category", ""),
        "department": item.get("department", ""),
        "brand":      item.get("brand", ""),
        "price":      item.get("price", 0),
        "images":     image_urls,
        "image_url":  image_urls[0] if image_urls else "",
    }


def build_product_page_content(item: dict) -> str:
    """
    Text dùng để embedding/search. Format có nhãn rõ ràng giúp model nhận biết
    đâu là tên, danh mục, màu sắc, chất liệu, dịp mặc, mô tả.
    """
    details = item.get("details", {}) or {}
    fields = [
        ("Tên sản phẩm", item.get("title")),
        ("Mã sản phẩm", item.get("product_id")),
        ("Danh mục", item.get("category")),
        ("Đối tượng", item.get("department")),
        ("Thương hiệu", item.get("brand")),
        ("Giá", f"{item.get('price', 0)} VND"),
        ("Màu sắc", details.get("main_color")),
        ("Chất liệu", details.get("material")),
        ("Kích cỡ", details.get("size")),
        ("Họa tiết", details.get("pattern")),
        ("Mùa phù hợp", item.get("season")),
        ("Dịp sử dụng", item.get("occasion")),
        ("Mô tả", item.get("description")),
    ]
    return "\n".join(f"{label}: {normalize_to_text(value)}" for label, value in fields)


def process_fashion_metadata(file_path: str | Path) -> list:
    """
    Đọc 1 file JSONL và chuyển từng sản phẩm thành LangChain Document.
    File 65k đã có đầy đủ category/brand/price/images nên không cần tách thành nhiều file.
    """
    file_path = Path(file_path)
    documents = []
    stats = {
        "total_lines": 0,
        "json_errors": 0,
        "missing_product_id": 0,
        "missing_category": 0,
        "missing_image": 0,
    }

    with file_path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            stats["total_lines"] += 1
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                stats["json_errors"] += 1
                continue

            metadata = build_product_metadata(item)
            if not metadata["product_id"]:
                stats["missing_product_id"] += 1
            if not metadata["category"]:
                stats["missing_category"] += 1
            if not metadata["image_url"]:
                stats["missing_image"] += 1

            documents.append(Document(
                page_content=build_product_page_content(item),
                metadata=metadata,
            ))

    print(
        f"[THỐNG KÊ] dòng={stats['total_lines']} | docs={len(documents)} | "
        f"lỗi_json={stats['json_errors']} | "
        f"thiếu_id={stats['missing_product_id']} | "
        f"thiếu_category={stats['missing_category']} | "
        f"thiếu_ảnh={stats['missing_image']}"
    )
    return documents


def run_data_pipeline(metadata_file: str | Path = CHATBOT_FASHION_DIR / "data" / "metadata" / "meta_Amazon_Lazada_Fashion_65k.jsonl",
                      qdrant_url: str = "http://localhost:6333",
                      collection_name: str = "fashion_products_vifashionclip_vi_65k_structured_vi"):
    """
    Hàm tổng: đọc 1 file JSONL 65k → embed bằng ViFashionCLIP → đẩy vào Qdrant.
    Có resume theo số point hiện có trong collection, nên file đầu vào cần giữ thứ tự ổn định.
    """
    metadata_file = Path(metadata_file)
    if not metadata_file.exists():
        print(f"[LỖI] Không tìm thấy file metadata: {metadata_file}")
        return

    print(f"[THÔNG BÁO] Đọc dữ liệu từ file: {metadata_file}")
    all_docs = process_fashion_metadata(metadata_file)
    print(f"[OK] Tổng số Documents: {len(all_docs)}")

    emb    = ViFashionCLIPTextEmbeddings()
    client = QdrantClient(url=qdrant_url)

    if not client.collection_exists(collection_name):
        print(f"[THÔNG BÁO] Tạo mới collection '{collection_name}'...")
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=512, distance=Distance.COSINE),
        )
        current_count = 0
    else:
        current_count = client.get_collection(collection_name).points_count
        print(f"[THÔNG BÁO] Collection đã có {current_count} sản phẩm.")

    if current_count > len(all_docs):
        print("[LỖI] Collection có nhiều point hơn file metadata. Hãy dùng collection mới để tránh trộn dữ liệu.")
        return

    remaining = all_docs[current_count:]
    if not remaining:
        print("[OK] Dữ liệu đã được index đầy đủ. Không cần chạy lại!")
        return

    print(f"[THÔNG BÁO] Bắt đầu index {len(remaining)} sản phẩm còn lại...")
    vdb = QdrantVectorStore(client=client, collection_name=collection_name, embedding=emb)

    batch_size = 128
    with tqdm(total=len(all_docs), initial=current_count,
              desc="Tiến độ index", unit="SP") as pbar:
        for i in range(0, len(remaining), batch_size):
            batch = remaining[i : i + batch_size]
            vdb.add_documents(documents=batch)
            pbar.update(len(batch))

    print("[OK] Index hoàn tất vào Qdrant!")


# ⚠️ Bỏ comment dòng dưới NẾU muốn chạy index file 65k.
#    Cẩn thận: việc này mất nhiều thời gian và tài nguyên!
# run_data_pipeline()

print("[OK] Các hàm Data Pipeline đã sẵn sàng.")
print("     Gọi run_data_pipeline() để bắt đầu index dữ liệu.")


[OK] Các hàm Data Pipeline đã sẵn sàng.
     Gọi run_data_pipeline() để bắt đầu index dữ liệu.


## PHẦN 4: Kết nối Qdrant & Khởi tạo Vector Store

Cell này tạo 2 embedding objects, kết nối Qdrant, và khởi tạo retriever với **diversity filter** (top-10 → lọc đa dạng → tối đa 3 sản phẩm).

In [7]:
print("[THÔNG BÁO] Đang khởi tạo embedding models...")

# Layer A (sản phẩm): ViFashionCLIP tiếng Việt, vector 512 chiều
product_embeddings = ViFashionCLIPTextEmbeddings()

# Layer B (quy tắc phối đồ): BGE-M3, vector 1024 chiều
rule_embeddings = BGEM3Embeddings()

print("[THÔNG BÁO] Đang kết nối Qdrant...")
# ── Chọn 1 trong 2 cách kết nối bên dưới ──
# [A] Local file (không cần Docker) — đang dùng
client = QdrantClient(path="../qdrant_data_65k/qdrant_data")
# [B] Docker server (chạy docker-compose up trước)
# client = QdrantClient(url="http://localhost:6333")

PRODUCT_COLLECTION = "fashion_products_vifashionclip_vi_65k_structured_vi"

vector_db = QdrantVectorStore(
    client=client,
    collection_name=PRODUCT_COLLECTION,
    embedding=product_embeddings,
)


def normalize_product_metadata(doc: Document) -> Document:
    """Đảm bảo mỗi sản phẩm có một image_url chuẩn để render trong prompt."""
    images = doc.metadata.get("images", [])
    if isinstance(images, str):
        images = [images] if images else []
    doc.metadata["image_url"] = doc.metadata.get("image_url") or (images[0] if images else "")
    return doc


def diversity_filter_documents(docs: list[Document], max_docs: int = 3) -> list[Document]:
    """Giữ tối đa một sản phẩm cho mỗi brand và category sau khi retrieve từ Qdrant."""
    selected, seen_brands, seen_categories = [], set(), set()
    for doc in docs:
        doc = normalize_product_metadata(doc)
        brand    = str(doc.metadata.get("brand", "")).strip().lower()
        category = str(doc.metadata.get("category", "")).strip().lower()
        if brand and brand in seen_brands:
            continue
        if category and category in seen_categories:
            continue
        selected.append(doc)
        if brand:
            seen_brands.add(brand)
        if category:
            seen_categories.add(category)
        if len(selected) >= max_docs:
            break
    return selected


class DiversityFilteredRetriever(BaseRetriever):
    """Retriever bọc diversity filter: chọn top-k rồi lọc đa dạng brand/category."""
    base_retriever: Any
    max_docs: int = 3

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> list[Document]:
        raw_docs = self.base_retriever.invoke(query)
        return diversity_filter_documents(raw_docs, max_docs=self.max_docs)


# Lấy top-10 ứng viên, sau đó chỉ đưa tối đa 3 sản phẩm đa dạng vào LLM
base_retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10},
)
retriever = DiversityFilteredRetriever(base_retriever=base_retriever, max_docs=3)

print(f"[OK] Đã kết nối Qdrant (local file)")
print(f"[OK] Collection sản phẩm : {PRODUCT_COLLECTION} (512-dim ViFashionCLIP)")
print(f"[OK] Retriever sẵn sàng : top-10 → lọc đa dạng → tối đa 3 sản phẩm")


[THÔNG BÁO] Đang khởi tạo embedding models...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[OK] ViFashionCLIP đã tải: d:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion\Vietnamese\vifashionclip_aiteamvn_embedding_v2_projection_336k\stage2_last_layers\best_stage2_model.pt
     Device: cpu | Embedding dim: 512
[THÔNG BÁO] Đang kết nối Qdrant...


C:\Users\DELL\AppData\Local\Temp\ipykernel_27780\1425617383.py:12: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <fashion_products_vifashionclip_vi_65k_structured_vi> contains 65480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path="../qdrant_data_65k/qdrant_data")


[OK] Đã kết nối Qdrant (local file)
[OK] Collection sản phẩm : fashion_products_vifashionclip_vi_65k_structured_vi (512-dim ViFashionCLIP)
[OK] Retriever sẵn sàng : top-10 → lọc đa dạng → tối đa 3 sản phẩm


## PH?N 4.5: Image Retrieval ? Index & Search ?nh MAIN b?ng FashionCLIP

> Nh?nh n?y t?o collection ?nh ri?ng: m?i s?n ph?m l?y **1 ?nh MAIN** ? encode b?ng FashionCLIP image encoder ? l?u vector 512 chi?u v?o Qdrant.


In [ ]:
PRODUCT_IMAGE_COLLECTION = "fashion_products_fashionclip_image_main_65k"
PRODUCT_IMAGE_ROOT = CHATBOT_FASHION_DIR.parent / "Amazon_Lazada_Fashion_Metadata_65k" / "images"

# Lazy-load to avoid using RAM/GPU when you only chat with text.
image_embeddings = None


def get_image_embeddings() -> FashionCLIPImageEmbeddings:
    global image_embeddings
    if image_embeddings is None:
        image_embeddings = FashionCLIPImageEmbeddings(batch_size=32)
    return image_embeddings


def get_main_image_relative_path(item: dict) -> str:
    """Get MAIN image from metadata. Fallback to the first image if MAIN is missing."""
    images = item.get("images", []) or []

    for img in images:
        if str(img.get("variant", "")).upper() == "MAIN" and img.get("large"):
            return img["large"]

    for img in images:
        large = str(img.get("large", ""))
        if "_MAIN" in large.upper():
            return large

    if images and images[0].get("large"):
        return images[0]["large"]

    product_id = item.get("product_id")
    return f"images/{product_id}_MAIN.jpg" if product_id else ""


def resolve_main_image_path(relative_path: str, image_root: str | Path = PRODUCT_IMAGE_ROOT) -> Path:
    """Convert metadata path like images/ABC_MAIN.jpg to the local image path."""
    rel = Path(str(relative_path).replace("\\", "/"))
    if rel.parts and rel.parts[0].lower() == "images":
        rel = Path(*rel.parts[1:])
    return Path(image_root) / rel


def iter_products_with_main_image(metadata_file: str | Path,
                                  image_root: str | Path = PRODUCT_IMAGE_ROOT):
    """Yield products whose MAIN image exists locally."""
    metadata_file = Path(metadata_file)
    image_root = Path(image_root)

    with metadata_file.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue

            rel_path = get_main_image_relative_path(item)
            img_path = resolve_main_image_path(rel_path, image_root)
            if not img_path.exists():
                continue

            yield item, rel_path, img_path


def build_image_payload(item: dict, rel_path: str, img_path: Path) -> dict:
    """Payload stored with the image vector so the LLM can answer with product details."""
    metadata = build_product_metadata(item)
    metadata["main_image_path"] = str(img_path)
    metadata["main_image_relpath"] = rel_path
    metadata["image_url"] = metadata.get("image_url") or rel_path

    return {
        "product_id": metadata.get("product_id", ""),
        "title": metadata.get("title", ""),
        "category": metadata.get("category", ""),
        "department": metadata.get("department", ""),
        "brand": metadata.get("brand", ""),
        "price": metadata.get("price", 0),
        "image_url": metadata.get("image_url", ""),
        "main_image_path": str(img_path),
        "main_image_relpath": rel_path,
        "page_content": build_product_page_content(item),
        "metadata": metadata,
    }


def run_main_image_index_pipeline(
    metadata_file: str | Path = CHATBOT_FASHION_DIR / "data" / "metadata" / "meta_Amazon_Lazada_Fashion_65k.jsonl",
    image_root: str | Path = PRODUCT_IMAGE_ROOT,
    collection_name: str = PRODUCT_IMAGE_COLLECTION,
    batch_size: int = 64,
):
    """
    Index 1 MAIN image per product into Qdrant with FashionCLIP image vectors.
    Resume is based on current point count, so keep the metadata order stable.
    """
    metadata_file = Path(metadata_file)
    image_root = Path(image_root)

    if not metadata_file.exists():
        print(f"[ERROR] Metadata file not found: {metadata_file}")
        return
    if not image_root.exists():
        print(f"[ERROR] Image folder not found: {image_root}")
        return

    print(f"[INFO] Scanning MAIN images in: {image_root}")
    items = list(iter_products_with_main_image(metadata_file, image_root))
    print(f"[OK] Found {len(items)} products with local MAIN images")

    if not client.collection_exists(collection_name):
        print(f"[INFO] Creating image collection: {collection_name}")
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=512, distance=Distance.COSINE),
        )
        current_count = 0
    else:
        current_count = client.count(collection_name).count
        print(f"[INFO] Image collection already has {current_count} vectors")

    if current_count > len(items):
        print("[ERROR] Collection has more points than available MAIN images. Use a new collection name to avoid mixed data.")
        return

    remaining = items[current_count:]
    if not remaining:
        print("[OK] MAIN images are already fully indexed")
        return

    emb = get_image_embeddings()
    print(f"[INFO] Indexing {len(remaining)} remaining MAIN images...")

    with tqdm(total=len(items), initial=current_count, desc="Image index progress", unit="img") as pbar:
        for start in range(0, len(remaining), batch_size):
            batch = remaining[start:start + batch_size]
            image_paths = [img_path for _, _, img_path in batch]
            vectors = emb.encode_image_paths(image_paths)

            points = []
            for (item, rel_path, img_path), vector in zip(batch, vectors):
                if vector is None:
                    continue
                product_id = str(item.get("product_id", ""))
                point_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f"fashion-main-image:{product_id}"))
                points.append(PointStruct(
                    id=point_id,
                    vector=vector,
                    payload=build_image_payload(item, rel_path, img_path),
                ))

            if points:
                client.upsert(collection_name=collection_name, points=points)
            pbar.update(len(batch))

    print(f"[OK] MAIN image indexing completed -> {collection_name}")


def image_point_to_document(point) -> Document:
    """Convert a Qdrant image point back to a LangChain Document for the existing prompt."""
    payload = point.payload or {}
    metadata = payload.get("metadata", {}) or {}
    metadata["image_search_score"] = getattr(point, "score", None)
    metadata["image_url"] = payload.get("image_url") or metadata.get("image_url", "")
    return Document(
        page_content=payload.get("page_content", ""),
        metadata=metadata,
    )


def search_products_by_image(image_path: str | Path,
                             collection_name: str = PRODUCT_IMAGE_COLLECTION,
                             top_k: int = 12,
                             max_products: int = 3,
                             score_threshold: float | None = 0.15) -> list[Document]:
    """Search products by FashionCLIP image vector, grouped to max 1 result per product_id."""
    image_path = Path(image_path)
    if not image_path.exists():
        print(f"[ERROR] Query image not found: {image_path}")
        return []
    if not client.collection_exists(collection_name):
        print(f"[WARN] Image collection not found: {collection_name}. Run run_main_image_index_pipeline() first.")
        return []

    query_vector = get_image_embeddings().embed_image(image_path)
    if query_vector is None:
        return []

    kwargs = {
        "collection_name": collection_name,
        "query": query_vector,
        "limit": top_k,
        "with_payload": True,
    }
    if score_threshold is not None:
        kwargs["score_threshold"] = score_threshold

    response = client.query_points(**kwargs)

    docs, seen_product_ids = [], set()
    for point in response.points:
        doc = image_point_to_document(point)
        product_id = doc.metadata.get("product_id", "")
        if product_id and product_id in seen_product_ids:
            continue
        docs.append(normalize_product_metadata(doc))
        if product_id:
            seen_product_ids.add(product_id)
        if len(docs) >= max_products:
            break

    return docs


# Run this once when you want to build the MAIN-image collection.
# run_main_image_index_pipeline()

print("[OK] Image Retrieval ready: MAIN image -> FashionCLIP 512-dim -> Qdrant")
print(f"     Image collection: {PRODUCT_IMAGE_COLLECTION}")
print(f"     Image folder    : {PRODUCT_IMAGE_ROOT}")


## PHẦN 5: Layer B — Nạp & Index kiến thức phối đồ

In [8]:
def load_layer_b(file_path: str) -> list:
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

layer_b_female = load_layer_b(CHATBOT_FASHION_DIR / "data" / "stylists" / "Layer_B_Female_Knowledge.json")
layer_b_male   = load_layer_b(CHATBOT_FASHION_DIR / "data" / "stylists" / "Layer_B_Male_Knowledge.json")
print(f"[OK] Layer B: {len(layer_b_female)} rules Nữ | {len(layer_b_male)} rules Nam")


[OK] Layer B: 880 rules Nữ | 416 rules Nam


In [9]:
def index_layer_b(data: list, collection_name: str):
    """
    Index toàn bộ Layer B vào Qdrant bằng BGE-M3 (chạy 1 lần).
    Dùng rule_embeddings (global) để tránh phụ thuộc vào vector_db.embeddings.
    """
    existing = [c.name for c in client.get_collections().collections]
    if collection_name in existing:
        count = client.count(collection_name).count
        print(f"[SKIP] {collection_name} đã tồn tại ({count} rules) — bỏ qua index.")
        return

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )

    points = []
    for i, rule in enumerate(data):
        # Ghép các field có ngữ nghĩa để embed — bge-m3 xử lý tốt đa ngôn ngữ Vi+En
        text   = (f"{rule['rule_key']} {rule['phong_cach']} "
                  f"{rule['boi_canh']} {rule['ly_do_tu_van']}")
        vector = rule_embeddings.embed_query(text)   # dùng rule_embeddings trực tiếp
        points.append(PointStruct(id=i, vector=vector, payload=rule))

    client.upsert(collection_name=collection_name, points=points)
    print(f"[OK] Indexed {len(points)} rules → {collection_name}")

index_layer_b(layer_b_female, "layer_b_female")
index_layer_b(layer_b_male,   "layer_b_male")


[SKIP] layer_b_female đã tồn tại (880 rules) — bỏ qua index.
[SKIP] layer_b_male đã tồn tại (416 rules) — bỏ qua index.


In [10]:
def find_matching_rule(user_query: str,
                       gender: str  = "female",
                       profile: dict = None) -> dict | None:
    """
    Tìm quy tắc phối đồ phù hợp nhất từ Layer B.

    Chiến lược fallback 3 bước:
      1. Lọc theo cả Dáng người + Tone da
      2. Nới lọc: chỉ giữ Dáng người
      3. Bỏ hết bộ lọc, lấy kết quả gần giống nhất
    """
    collection   = f"layer_b_{gender}"
    query_vector = rule_embeddings.embed_query(user_query)

    conditions = []
    if profile:
        if profile.get("dang_nguoi"):
            conditions.append(FieldCondition(
                key="dang_nguoi",
                match=MatchAny(any=[profile["dang_nguoi"], "Mọi vóc dáng"])
            ))
        if profile.get("tone_da"):
            conditions.append(FieldCondition(
                key="tone_da",
                match=MatchAny(any=[profile["tone_da"], "Mọi tone da"])
            ))

    # BƯỚC 1: Lọc khắt khe cả Dáng + Da
    search_filter = Filter(must=conditions) if conditions else None
    response = client.query_points(
        collection_name=collection, query=query_vector,
        query_filter=search_filter, limit=1, score_threshold=0.50,
    )
    results = response.points

    # BƯỚC 2 (FALLBACK): Bỏ tiêu chí Da, chỉ giữ tiêu chí Dáng
    if not results and profile and profile.get("tone_da") and profile.get("dang_nguoi"):
        fallback_conditions = [
            FieldCondition(key="dang_nguoi", match=MatchAny(any=[profile["dang_nguoi"], "Mọi vóc dáng"]))
        ]
        search_filter = Filter(must=fallback_conditions)
        response = client.query_points(
            collection_name=collection, query=query_vector,
            query_filter=search_filter, limit=1, score_threshold=0.50,
        )
        results = response.points

    # BƯỚC 3 (FALLBACK): Bỏ sạch bộ lọc để lấy kết quả gần giống nhất
    if not results and search_filter:
        response = client.query_points(
            collection_name=collection, query=query_vector,
            limit=1, score_threshold=0.50,
        )
        results = response.points

    return results[0].payload if results else None


def find_outfit_details(base_rule: dict, gender: str = "female") -> dict:
    """
    Từ công thức chủ đạo (đã tìm ở find_matching_rule), tìm chi tiết
    từng món đồ cần phối (VD: Áo kiểu gì, Quần kiểu gì).

    Chiến lược:
      1. Tìm khớp chính xác bằng chữ (nhanh, không cần Qdrant)
      2. Fallback: Semantic search qua Qdrant nếu không khớp
    """
    collection   = f"layer_b_{gender}"
    outfit_rules = {}
    style_query  = f"{base_rule['phong_cach']} {base_rule['boi_canh']}"

    for category in base_rule.get("goi_y_phoi_cung", []):

        # CÁCH 1: Tìm khớp chính xác bằng chữ (không cần Qdrant)
        exact = [r for r in (layer_b_female if gender == "female" else layer_b_male)
                 if r["rule_key"].startswith(category)
                 and r["phong_cach"] == base_rule["phong_cach"]
                 and r["boi_canh"]   == base_rule["boi_canh"]]

        if exact:
            outfit_rules[category] = exact[0]
            continue

        # CÁCH 2 (FALLBACK): Semantic search qua Qdrant
        query_vector = rule_embeddings.embed_query(f"{category} {style_query}")

        raw_response = client.query_points(
            collection_name=collection, query=query_vector,
            limit=10, score_threshold=0.40,
        )

        # Lọc chỉ giữ những rule đúng category cần tìm
        category_results = [r for r in raw_response.points
                            if r.payload["rule_key"].startswith(category)]

        if category_results:
            outfit_rules[category] = category_results[0].payload

    return outfit_rules

print("[OK] Các hàm Layer B đã sẵn sàng!")


[OK] Các hàm Layer B đã sẵn sàng!


## PHẦN 6: Category Mapping — Layer B → Layer A

In [11]:
# TỪ ĐIỂN CƠ BẢN: Dịch tên category từ Layer B sang tên kệ hàng Layer A
# Ví dụ: Stylist gọi "Áo mặc trong" → Thủ kho gọi là kệ "Áo"
CATEGORY_MAPPING = {
    "Áo mặc trong (áo thun/sơ mi)": ["Áo"],
    "Áo khoác ngoài":               ["Áo khoác"],
    "Áo khoác nhẹ/Áo len":          ["Áo khoác"],
    "Quần/Chân váy":                 ["Quần", "Chân váy"],
    "Đầm/Jumpsuit":                  ["Đầm", "Jumpsuit"],
    "Giày dép":                      ["Giày"],
    "Túi xách":                      ["Túi xách"],
    "Phụ kiện":                      None,  # Chung chung → dùng từ điển nâng cao bên dưới
}

# TỪ ĐIỂN NÂNG CAO: Soi từ khoá tiếng Anh để xác định Phụ Kiện cụ thể nằm ở kệ nào
PHU_KIEN_KEYWORD_ROUTER = {
    "Mũ":             ["beret","hat","cap","beanie","fedora","bucket","brim", "flat cap","earflap","snood","balaclava","trapper"],
    "Găng tay":        ["gloves","glove","arm warmer"],
    "Kính mắt":        ["glasses","sunglasses","sunglass"],
    "Đồng hồ":         ["watch"],
    "Dây chuyền":      ["necklace","chain pendant","chain"],
    "Bông tai":        ["earring","earrings"],
    "Vòng tay":        ["bracelet"],
    "Nhẫn":            ["ring"],
    "Ghim cài áo":     ["brooch","pin","badge"],
    "Phụ kiện hỗ trợ": ["socks","sock","scarf","tie","belt","bandana", "headband","suspender"],
}

def get_layer_a_categories(layer_b_category: str, product_type: str) -> list:
    """Phiên dịch tên category từ Layer B sang tên kệ hàng Layer A."""
    # Nếu không phải Phụ kiện → dùng ngay Từ điển Cơ Bản
    if layer_b_category != "Phụ kiện":
        return CATEGORY_MAPPING.get(layer_b_category, [])
    
    # Nếu là Phụ kiện → soi từ khoá tiếng Anh bằng Từ điển Nâng Cao
    ptype_lower = product_type.lower()
    for layer_a_cat, keywords in PHU_KIEN_KEYWORD_ROUTER.items():
        if any(kw in ptype_lower for kw in keywords):
            return [layer_a_cat]  # VD: có chữ "watch" → trả về kệ "Đồng hồ"
            
    # Tra không ra thì xếp vào kệ "Phụ kiện hỗ trợ"
    return ["Phụ kiện hỗ trợ"]


def get_products_for_outfit(product_type: str, layer_b_category: str,
                            phong_cach: str, vdb) -> list:
    """
    Tìm sản phẩm Layer A phù hợp cho từng món đồ trong outfit.
    Có ngưỡng điểm (threshold) để lọc đồ không liên quan.
    """
    
    # BƯỚC 1: Phiên dịch tên kệ hàng
    target_categories = get_layer_a_categories(layer_b_category, product_type)
    
    search_filter = None
    if target_categories:
        search_filter = Filter(must=[
            FieldCondition(
                key="metadata.category",
                match=MatchAny(any=target_categories)
            )
        ])
        
    query = f"{product_type} {phong_cach}"
    
    # BƯỚC 2: Tìm kiếm với điểm độ tương đồng (cosine similarity)
    raw_results = vdb.similarity_search_with_score(query=query, k=8, filter=search_filter)
    
    # BƯỚC 3: Lọc theo ngưỡng 0.20 (ngưỡng phù hợp cho ViFashionCLIP 512-dim)
    valid_products = []
    for doc, score in raw_results:
        if score >= 0.20:
            valid_products.append(normalize_product_metadata(doc))
            
    return diversity_filter_documents(valid_products, max_docs=3)


## PHẦN 7: Xử lý ảnh với Qwen2.5-VL
> Yêu cầu: `ollama pull qwen2.5vl:3b` trước khi chạy phần này.

In [12]:
VL_MAX_SIZE = 512   # px — giảm kích thước ảnh để tránh lỗi GGML_ASSERT

# Giá trị dáng_người + tone_da khớp với Layer B
LAYER_B_DANG_NGUOI = [
    "Dáng quả lê", "Dáng quả táo", "Dáng đồng hồ cát",
    "Dáng chữ nhật", "Dáng cân đối",
    "Người thấp bé", "Người ngoại cỡ", "Người mảnh",
]
LAYER_B_TONE_DA = [
    "Da sáng", "Da trung bình", "Da ngăm", "Da ấm",
]

LAYER_B_WILDCARD_DANG = "Mọi vóc dáng"
LAYER_B_WILDCARD_TONE = "Mọi tone da"


# ── 1. Tiền xử lý ảnh ─────────────────────────────────────────────────
def _preprocess_image(image_path: str) -> str:
    """
    Resize ảnh về VL_MAX_SIZE (giữ tỉ lệ) và convert sang JPEG.
    Giải quyết lỗi GGML_ASSERT do Vision encoder Qwen-VL có giới hạn
    số patch token — ảnh lớn (>1024px) vượt giới hạn gây crash.
    """
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    if max(w, h) > VL_MAX_SIZE:
        ratio   = VL_MAX_SIZE / max(w, h)
        img     = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()


# ── 2. Gọi VL model ────────────────────────────────────────────────
def _call_vl(image_path: str, prompt: str) -> str:
    """
    Gọi Qwen2.5-VL qua Ollama.
    Ảnh được resize + convert JPEG trước để tránh lỗi GGML.
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")
    img_b64 = _preprocess_image(image_path)
    try:
        resp = ollama.chat(
            model    = QWEN_VL_MODEL,
            messages = [{"role": "user", "content": prompt, "images": [img_b64]}],
        )
        return resp["message"]["content"].strip()
    except Exception as e:
        print(f"   ⚠️  [LỖI VISION] {e}")
        return ""


# ── 3. Phân loại ảnh ───────────────────────────────────────────────
def detect_image_type(image_path: str, user_query: str = "") -> str:
    """
    Xác định mục đích gửi ảnh dựa vào ảnh + câu hỏi đi kèm.
    Ưu tiên PRODUCT nếu user hỏi về sản phẩm (dù ảnh có người mẫu).
    """
    prompt = f"""Ảnh này chứa người hay chứa sản phẩm?
Câu hỏi của người dùng: "{user_query}"

Dựa vào câu hỏi và ảnh, xác định MỤC ĐÍCH của người dùng:
1. Muốn tìm/hỏi về quần áo/món đồ trong ảnh (dù có người mẫu mặc) → PRODUCT
2. Muốn phân tích vóc dáng/tone da của người trong ảnh             → PERSON
3. Không có câu hỏi + ảnh chủ yếu là người                        → PERSON
4. Không có câu hỏi + ảnh chụp sát sản phẩm                       → PRODUCT

Chỉ trả lời đúng 1 chữ: PERSON hoặc PRODUCT."""
    result = _call_vl(image_path, prompt).upper()
    return "product" if "PRODUCT" in result else "person"


# ── 4. Phân tích người ──────────────────────────────────────────────
def analyze_person_image(image_path: str) -> dict:
    """
    Phân tích dáng người + tone da.
    Prompt dùng đúng các giá trị có trong Layer B để filter khớp.
    """
    dang_list = " | ".join(LAYER_B_DANG_NGUOI)
    tone_list = " | ".join(LAYER_B_TONE_DA)

    prompt = f"""Bạn là chuyên gia tư vấn thời trang. Phân tích người trong ảnh:

1. DÁNG NGƯỜI (chọn đúng 1 trong danh sách): {dang_list}
2. TONE DA (chọn đúng 1 trong danh sách): {tone_list}
3. NHẬN XÉT: 1-2 câu về điểm nổi bật có thể khai thác khi phối đồ.

Trả lời theo đúng format (không thêm gì khác):
DÁNG: [tên dáng]
TONE: [tên tone]
NHẬN XÉT: [nội dung]"""

    raw     = _call_vl(image_path, prompt)
    profile = {"dang_nguoi": None, "tone_da": None, "nhan_xet": ""}

    for line in raw.splitlines():
        line = line.strip()
        upper = line.upper()
        if "DÁNG" in upper and ":" in line:
            val = line.split(":", 1)[1].strip()
            profile["dang_nguoi"] = val if val else None
        elif "TONE" in upper and ":" in line:
            val = line.split(":", 1)[1].strip()
            profile["tone_da"] = val if val else None
        elif "NHẬN XÉT" in upper and ":" in line:
            profile["nhan_xet"] = line.split(":", 1)[1].strip()

    return profile


# ── 5. Caption sản phẩm ─────────────────────────────────────────────
def caption_product_image(image_path: str, user_query: str = "") -> str:
    """
    Mô tả món đồ thời trang trong ảnh bằng tiếng Việt.
    user_query giúp VL tập trung đúng vào món đồ user đang hỏi.
    """
    prompt = f"""Câu hỏi của người dùng: "{user_query}"

Mô tả chi tiết MÓN ĐỒ THỜI TRANG mà người dùng đang quan tâm trong ảnh bằng tiếng Việt.
Bao gồm: loại sản phẩm, màu sắc, kiểu dáng, chất liệu (nếu nhận ra), phong cách.
Ngắn gọn 1-2 câu. TUYỆT ĐỐI KHÔNG mô tả người mẫu."""
    return _call_vl(image_path, prompt)


print(f"[OK] Các hàm Vision đã sẵn sàng!")
print(f"     Model   : {QWEN_VL_MODEL}")
print(f"     Max size: {VL_MAX_SIZE}px (tự động resize → tránh lỗi GGML_ASSERT)")
print(f"     Dáng    : {len(LAYER_B_DANG_NGUOI)} giá trị khớp Layer B")
print(f"     Tone    : {len(LAYER_B_TONE_DA)} giá trị khớp Layer B")


[OK] Các hàm Vision đã sẵn sàng!
     Model   : qwen2.5vl:3b
     Max size: 512px (tự động resize → tránh lỗi GGML_ASSERT)
     Dáng    : 8 giá trị khớp Layer B
     Tone    : 4 giá trị khớp Layer B


## PHẦN 8: Khởi tạo LLM

In [13]:
print(f"[THÔNG BÁO] Đang khởi tạo LLM qua Ollama: {LLM_MODEL}")

llm = ChatOllama(
    model       = LLM_MODEL,
    temperature = 0.4,
    timeout     = 120,
    num_predict = 1024,
    num_ctx     = 8192,
)

print(f"[OK] LLM đã sẵn sàng! Model: {LLM_MODEL}")


[THÔNG BÁO] Đang khởi tạo LLM qua Ollama: qwen3:4b-instruct
[OK] LLM đã sẵn sàng! Model: qwen3:4b-instruct


## PHẦN 9: Cấu hình Prompt & Redis

In [14]:
# ── Prompt chính cho luồng SEARCH (schema cứng + chống hallucination) ──
system_prompt = """Bạn là chuyên viên tư vấn thời trang cao cấp, thân thiện và nói tiếng Việt tự nhiên.

QUY TẮc TỐI CAO:
1. Chỉ dùng thông tin có trong phần "DỮ LIỆU SẢN PHẨM". Không bịa tên, mã, giá, ảnh hoặc đặc điểm.
2. Giới thiệu tối đa 3 sản phẩm. Nếu context có nhiều hơn, chọn 3 sản phẩm phù hợp nhất.
3. Toàn bộ câu trả lời không vượt quá 400 từ.
4. Trước khi trả lời, tự kiểm tra xem sản phẩm có đúng nhu cầu, màu sắc/dịp mặc/chất liệu có hợp lý không. Không trình bày quá trình suy luận nội bộ.
5. Kết thúc bằng đúng 1 câu hỏi gợi mở để tiếp tục tư vấn.

SCHEMA BẮT BUỘC:
Mình gợi ý cho bạn tối đa 3 lựa chọn sau:

1. **Tên sản phẩm**
- Mã SP: [MÃ_SP]
- Giá: [GIÁ] VND
- Đặc điểm: [màu/chất liệu/kiểu dáng/dịp mặc nổi bật]
- Lý do phù hợp: [1 câu ngắn, dựa trên yêu cầu của khách]
- Ảnh: ![Sản phẩm]([IMAGE_URL])

Nếu không có ảnh, ghi: "Ảnh: Chưa có ảnh".
Nếu không có sản phẩm phù hợp trong context, xin lỗi ngắn gọn và hỏi khách có muốn đổi phong cách, ngân sách hoặc danh mục không.

DỮ LIỆU SẢN PHẨM:
{context}"""

QA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── Prompt viết lại câu hỏi (query rewriting) ──
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", """Nhiệm vụ của bạn là VIẾT LẠI CÂU HỊI.
Dựa vào lịch sử trò chuyện, hãy làm rõ nghĩa của câu hỏi mới nhất để nó có thể đứng độc lập mà ai đọc cũng hiểu được.

QUY TẮc SỐNG CÒN:
- TUYỆT ĐỐI KHÔNG TRẢ LỜI CÂU HỊI CỦA KHÁCH.
- CHỈ IN RA DUY NHẤT CÂU HỊI ĐÃ ĐƯỢC VIẾT LẠI. Không giải thích, không dạ thưa, cấm thêm chữ "Câu hỏi là:".
- Nếu câu hỏi đã quá rõ ràng rồi, hãy in lại y nguyên câu đó.

VÍ DỤ:
Khách: "Có màu khác không?" → BẠN CHỈ ĐƯỢC IN RA: "Áo thun đỏ ở trên có màu khác không?"
"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── Format document gửi vào LLM: mỗi sản phẩm chỉ có 1 ảnh chuẩn ──
doc_prompt = PromptTemplate.from_template(
    "\n[MÃ_SP: {product_id}]"
    "\nIMAGE_URL: {image_url}"
    "\nTHÔNG TIN CHI TIẾT: {page_content}\n"
)

# ── Prompt cho luồng OUTFIT (schema cứng + self-check ẩn) ──



def format_documents_for_llm(docs: list[Document]) -> str:
    """Format Documents into context text for the product prompt."""
    lines = []
    for doc in docs:
        doc = normalize_product_metadata(doc)
        lines.append(doc_prompt.format(
            product_id=doc.metadata.get("product_id", "N/A"),
            image_url=doc.metadata.get("image_url", ""),
            page_content=doc.page_content,
        ))
    return "\n".join(lines)

OUTFIT_SYSTEM_PROMPT = """Bạn là chuyên gia tạo dáng (Personal Stylist) chuyên nghiệp và tâm lý.

NHIỆM VỤ: Dựa vào "CÔNG THỨC PHỐI ĐỒ" và "SẢN PHẨM GỢI Ý" bên dưới, tạo một outfit hoàn chỉnh cho khách.

QUY TẮc:
1. Chỉ giới thiệu sản phẩm có trong "SẢN PHẨM GỢI Ý". Không thêm món ngoài context.
2. Tối đa 3 sản phẩm, không vượt quá 400 từ.
3. Trước khi trả lời, tự kiểm tra sự hài hòa màu sắc, bối cảnh sử dụng và vóc dáng/tone da nếu có. Không trình bày quá trình suy luận nội bộ.
4. Kết thúc bằng đúng 1 câu hỏi gợi mở để tiếp tục tư vấn.

SCHEMA BẮT BUỘC:
Mình phối cho bạn một set như sau:

1. **Tên sản phẩm**
- Mã SP: [MÃ_SP]
- Giá: [GIÁ] VND
- Đặc điểm: [màu/chất liệu/kiểu dáng nổi bật]
- Lý do phù hợp: [1 câu ngắn, gắn với công thức phối đồ]
- Ảnh: ![Sản phẩm]([IMAGE_URL])

Nếu không có ảnh, ghi: "Ảnh: Chưa có ảnh".

{outfit_context}"""

outfit_prompt = ChatPromptTemplate.from_messages([
    ("system", OUTFIT_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── Redis lưu lịch sử hội thoại ──
REDIS_URL = "redis://localhost:6379"

SUMMARIZE_PROMPT = """Tóm tắt cuộc hội thoại mua sắm thời trang sau thành 3-5 câu ngắn.
Giữ lại: sản phẩm đã hỏi, phong cách khách thích, thông tin vóc dáng/tone da (nếu có).
Bỏ qua: lời chào, câu xã giao.
Chỉ trả về đoạn tóm tắt, không thêm gì khác.

Hội thoại:
{history_text}"""

def summarize_history(messages: list) -> str:
    """Dùng LLM tóm tắt lịch sử hội thoại cũ."""
    history_text = "\n".join([
        f"{'Khách' if m.type == 'human' else 'Bot'}: {m.content[:300]}"
        for m in messages
    ])
    resp = ollama.chat(
        model   = LLM_MODEL,
        messages= [{"role": "user",
                    "content": SUMMARIZE_PROMPT.format(history_text=history_text)}],
        options = {"temperature": 0, "num_predict": 150}
    )
    return resp["message"]["content"].strip()


def get_message_history(session_id: str):
    from langchain_core.messages import SystemMessage

    history  = RedisChatMessageHistory(session_id, url=REDIS_URL)
    messages = history.messages

    # Dưới 8 message → giữ nguyên, chưa cần tóm tắt
    if len(messages) <= 8:
        return history

    # Trên 8 message → tóm tắt phần cũ, giữ 4 message gần nhất
    old_messages    = messages[:-4]
    recent_messages = messages[-4:]

    summary_text = summarize_history(old_messages)

    history.clear()
    history.add_message(SystemMessage(
        content=f"[TÓM TẮT HỘI THOẠI TRƯỚC]: {summary_text}"
    ))
    history.add_messages(recent_messages)

    return history

print("[OK] Prompt & Redis đã sẵn sàng: schema cứng + image_url + tóm tắt lịch sử!")


[OK] Prompt & Redis đã sẵn sàng: schema cứng + image_url + tóm tắt lịch sử!


## PHẦN 10: Lắp ráp RAG Pipeline

In [15]:
print("[THÔNG BÁO] Đang lắp ráp RAG Pipeline...")

# ── Chain 1: Luồng SEARCH (RAG đầy đủ) ────────────────────────────
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
document_chain = create_stuff_documents_chain(
    llm=llm, prompt=QA_PROMPT, document_prompt=doc_prompt
)
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

full_chat_chain = RunnableWithMessageHistory(
    rag_chain, get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

# ── Chain 2: Luồng OUTFIT (không dùng retriever — context đã build sẵn) ──
# -- Chain 1.5: IMAGE SEARCH (docs retrieved by FashionCLIP image vectors) --
image_search_llm_chain = QA_PROMPT | llm
image_search_chain_with_history = RunnableWithMessageHistory(
    image_search_llm_chain, get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

outfit_llm_chain = outfit_prompt | llm
outfit_chain_with_history = RunnableWithMessageHistory(
    outfit_llm_chain, get_message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("[OK] RAG Pipeline đã sẵn sàng!")

[THÔNG BÁO] Đang lắp ráp RAG Pipeline...
[OK] RAG Pipeline đã sẵn sàng!


## PHẦN 11: Intent Detection

In [16]:
# ═══════════════════════════════════════════════════════════════════
# PHẦN 11: INTENT DETECTION — Hybrid (Keyword + LLM)
# ═══════════════════════════════════════════════════════════════════
#
# Chiến lược 2 tầng:
#   Tầng 1 — Keyword "đảm bảo": xử lý ngay, không gọi LLM → nhanh
#   Tầng 2 — LLM classify: chỉ gọi khi câu mơ hồ, có thêm context
#             của bot message trước để hiểu đúng hơn
# ═══════════════════════════════════════════════════════════════════

# ── Tầng 1: Keyword list "đảm bảo" (không bao giờ nhầm intent) ──
DEFINITE_GREETING = [
    "xin chào", "hello", "hi bạn", "chào bạn", "hey", "alo",
    "chào buổi sáng", "chào buổi chiều",
]
DEFINITE_CHITCHAT = [
    "cảm ơn", "cảm on", "thank you", "thanks", "tạm biệt",
    "bye", "hẹn gặp lại", "bái bai",
]
DEFINITE_OUTFIT = [
    "phối đồ", "mix match", "mặc với gì", "mặc cùng gì",
    "kết hợp với gì", "phối với gì", "outfit cho",
    "gợi ý outfit", "tư vấn phối",
]
DEFINITE_SEARCH = [
    "còn hàng không", "còn size", "giá bao nhiêu", "mã sp",
    "có bán không", "tìm giúp", "cho xem", "shop có",
]

# Từ khóa giới tính nam — chỉ dùng từ rõ ràng, tránh false positive
# Lưu ý: "anh" bị loại khỏi list vì hay xuất hiện trong "anh ơi" (user nữ gọi bot)
MALE_KEYWORDS = ["nam", "con trai", "bạn trai", "chàng", "đàn ông", "bố", "chồng"]

# ── Tầng 2: LLM classify ──
INTENT_CLASSIFY_PROMPT = """Bạn là bộ phân loại intent cho chatbot tư vấn thời trang.
Phân loại câu hỏi vào đúng 1 trong 4 nhóm:

OUTFIT   → Hỏi cách phối đồ, mix-match, tư vấn mặc gì cho dịp/vóc dáng/phong cách
SEARCH   → Tìm sản phẩm cụ thể, hỏi giá, còn hàng, so sánh, xem ảnh sản phẩm
CHITCHAT → Cảm ơn, tạm biệt, hỏi thăm, câu xã giao không liên quan mua sắm
GREETING → Chào hỏi, bắt đầu cuộc trò chuyện

Ví dụ bắt buộc phải học thuộc:
"có áo nào mặc đi tiệc không?"          → SEARCH   (tìm sản phẩm, dù có từ "mặc")
"tìm đồ phù hợp với quần jean xanh"     → OUTFIT   (phối đồ, không phải tìm 1 sản phẩm đơn)
"tôi cao 1m55 nên mặc gì?"              → OUTFIT   (tư vấn theo vóc dáng)
"muốn trông thanh lịch hơn thì sao?"    → OUTFIT   (tư vấn phong cách)
"áo này với quần kia trông có hợp không?"→ OUTFIT   (hỏi về sự phối hợp)
"còn size L không?"                     → SEARCH   (hỏi tồn kho)
"oke cảm ơn bạn nhiều"                  → CHITCHAT
"đi làm"                                → OUTFIT   (trả lời câu hỏi "dịp nào" của bot)
{context_block}
Câu cần phân loại: "{query}"

Chỉ trả lời đúng 1 từ: OUTFIT / SEARCH / CHITCHAT / GREETING"""


def detect_intent_llm(query: str, last_bot_msg: str = "") -> str:
    """
    Dùng LLM phân loại intent cho những câu mơ hồ mà keyword không xử lý được.
    Truyền thêm last_bot_msg để LLM hiểu context (VD: user trả lời câu hỏi của bot).
    """
    context_block = ""
    if last_bot_msg:
        context_block = (
            f'\nContext — Bot vừa nói: "{last_bot_msg[:120]}..."\n'
        )

    try:
        resp = ollama.chat(
            model   = LLM_MODEL,
            messages= [{
                "role"   : "user",
                "content": INTENT_CLASSIFY_PROMPT.format(
                    query         = query,
                    context_block = context_block,
                )
            }],
            options = {
                "temperature": 0,        # deterministic — không cần sáng tạo
                "num_predict": 10,       # chỉ cần 1 từ → rất nhanh (~0.3-0.5s)
            }
        )
        result = resp["message"]["content"].strip().upper()

        for intent in ["OUTFIT", "SEARCH", "CHITCHAT", "GREETING"]:
            if intent in result:
                return intent.lower()

    except Exception as e:
        print(f"[WARN] LLM intent classify lỗi: {e} → fallback search")

    return "search"   # fallback an toàn


def detect_intent(query: str, last_bot_msg: str = "") -> str:
    """
    Hybrid intent detection:
      1. Keyword "đảm bảo" → trả về ngay (không gọi LLM, không tốn thời gian)
      2. Câu mơ hồ           → gọi LLM classify với context bot message trước
    """
    q = query.lower().strip()

    # Tầng 1: keyword đảm bảo (outfit/search ưu tiên hơn greeting/chitchat)
    if any(kw in q for kw in DEFINITE_OUTFIT):    return "outfit"
    if any(kw in q for kw in DEFINITE_SEARCH):    return "search"
    if any(kw in q for kw in DEFINITE_GREETING):  return "greeting"
    if any(kw in q for kw in DEFINITE_CHITCHAT):  return "chitchat"

    # Tầng 2: câu mơ hồ → LLM
    return detect_intent_llm(query, last_bot_msg)


def detect_gender(query: str) -> str:
    """
    Trả về 'male' nếu query đề cập rõ giới tính nam, mặc định 'female'.
    Lưu ý: từ 'đàn ông', 'nam', 'chàng'... mới trigger — không dùng 'anh'
    vì 'anh ơi' là cách user nữ gọi chatbot.
    """
    return "male" if any(kw in query.lower() for kw in MALE_KEYWORDS) else "female"


def get_greeting_response() -> str:
    return (
        "Xin chào! Mình là trợ lý tư vấn thời trang của shop. "
        "Bạn cần tìm sản phẩm hay muốn được gợi ý phối đồ hôm nay? 😊"
    )

def get_chitchat_response(query: str) -> str:
    return "Rất vui được hỗ trợ bạn! Bạn còn muốn hỏi thêm gì về thời trang không?"


# ── Kiểm tra nhanh ──
print("[OK] Intent Detection (Hybrid) đã sẵn sàng!")
print("     Tầng 1 (keyword): greeting / chitchat / outfit / search")
print("     Tầng 2 (LLM)    : xử lý câu mơ hồ + context bot message")

TEST_CASES = [
    ("xin chào bạn",                         "greeting"),
    ("cảm ơn bạn nhiều nha",                 "chitchat"),
    ("phối đồ với áo thun trắng",            "outfit"),
    ("còn size L không",                     "search"),
    ("tôi cao 1m55 nên mặc gì?",             "outfit → LLM"),
    ("có áo nào mặc đi tiệc không?",         "search → LLM"),
    ("muốn trông thanh lịch hơn thì sao?",   "outfit → LLM"),
    ("đi làm",                               "outfit → LLM (cần context)"),
]

print("\n--- Kiểm tra Tầng 1 (không gọi LLM) ---")
for query, expected in TEST_CASES:
    q = query.lower().strip()
    tier1 = None
    if any(kw in q for kw in DEFINITE_OUTFIT):    tier1 = "outfit"
    elif any(kw in q for kw in DEFINITE_SEARCH):  tier1 = "search"
    elif any(kw in q for kw in DEFINITE_GREETING): tier1 = "greeting"
    elif any(kw in q for kw in DEFINITE_CHITCHAT): tier1 = "chitchat"

    if tier1:
        status = "✅" if tier1 == expected.split(" →")[0] else "⚠️"
        print(f"  {status} [{tier1:8s}] ← keyword | '{query}'")
    else:
        print(f"  🔁 [ambiguous] → cần LLM  | '{query}' (expect: {expected})")


[OK] Intent Detection (Hybrid) đã sẵn sàng!
     Tầng 1 (keyword): greeting / chitchat / outfit / search
     Tầng 2 (LLM)    : xử lý câu mơ hồ + context bot message

--- Kiểm tra Tầng 1 (không gọi LLM) ---
  ✅ [greeting] ← keyword | 'xin chào bạn'
  ✅ [chitchat] ← keyword | 'cảm ơn bạn nhiều nha'
  ✅ [outfit  ] ← keyword | 'phối đồ với áo thun trắng'
  ✅ [search  ] ← keyword | 'còn size L không'
  🔁 [ambiguous] → cần LLM  | 'tôi cao 1m55 nên mặc gì?' (expect: outfit → LLM)
  🔁 [ambiguous] → cần LLM  | 'có áo nào mặc đi tiệc không?' (expect: search → LLM)
  🔁 [ambiguous] → cần LLM  | 'muốn trông thanh lịch hơn thì sao?' (expect: outfit → LLM)
  🔁 [ambiguous] → cần LLM  | 'đi làm' (expect: outfit → LLM (cần context))


## PHẦN 12: Xây dựng Outfit Context (Layer B → Layer A)

In [17]:
def build_outfit_context(user_query: str,
                         gender: str   = "female",
                         profile: dict = None) -> str:
    """
    Xâu chuỗi Layer B và Layer A thành hồ sơ outfit hoàn chỉnh để đưa vào LLM.
    Trả về chuỗi text được định dạng sẵn, hoặc chuỗi rỗng nếu không tìm thấy rule.
    """
    base_rule = find_matching_rule(user_query, gender, profile)
    if not base_rule:
        return ""

    outfit_rules = find_outfit_details(base_rule, gender)
    if not outfit_rules:
        return ""

    outfit_products = {}
    for layer_b_category, rule in outfit_rules.items():
        product_type = rule["rule_key"].split("|")[1].strip()
        products     = get_products_for_outfit(
            product_type, layer_b_category, base_rule["phong_cach"], vector_db
        )
        outfit_products[layer_b_category] = {
            "product_type": product_type,
            "ly_do":        rule["ly_do_tu_van"],
            "products":     products,
        }

    lines = [
        "CÔNG THỨC PHỐI ĐỒ:",
        f"  Phong cách: {base_rule['phong_cach']}",
        f"  Bối cảnh : {base_rule['boi_canh']}",
        f"  Lý do    : {base_rule['ly_do_tu_van']}",
    ]
    if profile and profile.get("dang_nguoi"):
        lines.append(f"  Dáng người: {profile['dang_nguoi']}")
    if profile and profile.get("tone_da"):
        lines.append(f"  Tone da   : {profile['tone_da']}")
    lines += ["", "SẢN PHẨM GỢI Ý:"]

    total_products = 0
    for cat, data in outfit_products.items():
        lines.append(f"\n[{cat} — {data['product_type']}]")
        lines.append(f"  Lý do: {data['ly_do']}")
        if data["products"]:
            for doc in data["products"]:
                if total_products >= 3:
                    break
                doc = normalize_product_metadata(doc)
                pid       = doc.metadata.get("product_id", "N/A")
                price_raw = doc.metadata.get("price", "N/A")
                image_url = doc.metadata.get("image_url", "")
                try:
                    price_fmt = f"{int(price_raw):,}".replace(",", ".")
                except Exception:
                    price_fmt = price_raw
                lines.append(f"  - (Mã SP: {pid} | Giá: {price_fmt} VND | IMAGE_URL: {image_url})")
                lines.append(f"    {doc.page_content[:600]}")
                total_products += 1
        else:
            lines.append("  - (Chưa có sản phẩm phù hợp trong kho)")
        if total_products >= 3:
            break

    return "\n".join(lines)

print("[OK] build_outfit_context sẵn sàng với tối đa 3 sản phẩm + image_url!")


[OK] build_outfit_context sẵn sàng với tối đa 3 sản phẩm + image_url!


## PHẦN 13: Luồng chạy chính (Chat Loop)

In [ ]:
SESSION_ID   = str(uuid.uuid4())
user_profile = {}  # Tích lũy dáng người + tone da qua từng lượt chat

MAX_QUERY_CHARS = 500
PROMPT_INJECTION_PATTERNS = [
    r"ignore\s+(all\s+)?previous",
    r"bo\s+qua\s+(moi\s+)?(huong\s+dan|lenh|prompt)",
    r"b\u1ecf\s+qua\s+(m\u1ecdi\s+)?(h\u01b0\u1edbng\s+d\u1eabn|l\u1ec7nh|prompt)",
    r"quen\s+(het|cac)\s+(huong\s+dan|lenh)",
    r"qu\u00ean\s+(h\u1ebft|c\u00e1c)\s+(h\u01b0\u1edbng\s+d\u1eabn|l\u1ec7nh)",
    r"dong\s+vai",
    r"\u0111\u00f3ng\s+vai",
    r"system\s+prompt",
    r"developer\s+message",
    r"tiet\s+lo\s+(prompt|huong\s+dan|lenh)",
    r"ti\u1ebft\s+l\u1ed9\s+(prompt|h\u01b0\u1edbng\s+d\u1eabn|l\u1ec7nh)",
]


def validate_user_query(query: str) -> tuple[bool, str]:
    """Chặn input quá dài hoặc prompt injection rõ ràng trước khi vào intent/RAG."""
    clean_query = query.strip()
    if len(clean_query) > MAX_QUERY_CHARS:
        return False, f"Tin nhắn hơi dài rồi ạ. Bạn rút gọn dưới {MAX_QUERY_CHARS} ký tự giúp mình nhé."
    lowered = clean_query.lower()
    if any(re.search(pattern, lowered, flags=re.IGNORECASE) for pattern in PROMPT_INJECTION_PATTERNS):
        return False, "Mình chỉ hỗ trợ tư vấn thời trang và sản phẩm trong shop. Bạn gửi lại nhu cầu mua sắm cụ thể giúp mình nhé."
    return True, ""


class SpyRetrieverHandler(BaseCallbackHandler):
    """Hiển thị câu hỏi sau khi LLM đã viết lại (dùng để debug)."""
    def on_retriever_start(self, serialized: dict, query: str, **kwargs):
        print(f"\n[DEBUG] Câu hỏi sau rewrite: {query}\n")


print("=" * 60)
print("  CHATBOT TƯ VẤN THỜI TRANG  ")
print("  Nhập '0' để thoát | Nhập đường dẫn ảnh nếu có")
print("=" * 60 + "\n")

while True:
    user_input = input("Bạn: ").strip()
    if user_input == "0":
        print("\nChatbot: Cảm ơn bạn đã ghé thăm shop. Hẹn gặp lại!")
        break
    if not user_input:
        continue

    is_valid, validation_message = validate_user_query(user_input)
    if not is_valid:
        print(f"Chatbot: {validation_message}")
        print("\n" + "-" * 60 + "\n")
        continue

    final_query = user_input

    raw_img = input("Ảnh (Enter để bỏ qua): ").strip()
    if raw_img:
        if not os.path.exists(raw_img):
            print(f"   Không tìm thấy file: {raw_img}")
        else:
            print("[Đang phân tích ảnh...]")
            image_type = detect_image_type(raw_img, user_input)
            print(f"   → Phát hiện: {image_type.upper()}")

            if image_type == "person":
                person_info = analyze_person_image(raw_img)
                if person_info["dang_nguoi"]:
                    user_profile["dang_nguoi"] = person_info["dang_nguoi"]
                if person_info["tone_da"]:
                    user_profile["tone_da"] = person_info["tone_da"]

                print(f"\nChatbot: Mình đã phân tích xong! "
                      f"Bạn có **{person_info['dang_nguoi']}** "
                      f"với **{person_info['tone_da']}**. "
                      f"{person_info['nhan_xet']} "
                      f"\n\nMình đã lưu thông tin này để tư vấn phối đồ phù hợp hơn. "
                      f"Bạn muốn gợi ý outfit cho dịp nào?")
                print("\n" + "-" * 60 + "\n")
                continue

            image_search_docs = search_products_by_image(raw_img, top_k=12, max_products=3)
            if image_search_docs:
                force_image_search = True
                final_query = f"Find products similar to the uploaded image. Extra request: {user_input}"
                print(f"   -> FashionCLIP found {len(image_search_docs)} nearest products")
            else:
                print("   -> Image vector search unavailable, fallback to Qwen-VL caption")
                caption = caption_product_image(raw_img, user_input)
                print(f"   -> Caption: {caption[:80]}...")
                final_query = f"{caption}. Extra request: {user_input}" if user_input else caption

    is_valid, validation_message = validate_user_query(final_query)
    if not is_valid:
        print(f"Chatbot: {validation_message}")
        print("\n" + "-" * 60 + "\n")
        continue

    intent         = "image_search" if force_image_search else detect_intent(final_query)
    current_gender = detect_gender(final_query)
    if current_gender == "male":
        user_profile["gender"] = "male"
    gender = user_profile.get("gender", current_gender)

    profile_status = (
        f"Giới: {gender.upper()} | "
        f"Dáng: {user_profile.get('dang_nguoi', '?')} | "
        f"Tone: {user_profile.get('tone_da', '?')}"
    ) if user_profile else "chưa có"

    print(f"[Intent: {intent.upper()} | Gender: {gender} | Profile: {profile_status}]")

    if intent == "greeting":
        print(f"Chatbot: {get_greeting_response()}")
        print("\n" + "-" * 60 + "\n")
        continue

    if intent == "chitchat":
        print(f"Chatbot: {get_chitchat_response(final_query)}")
        print("\n" + "-" * 60 + "\n")
        continue

    print("Chatbot: ", end="")
    start_time       = time.time()
    first_token_time = None

    try:
        if intent == "image_search":
            image_context = format_documents_for_llm(image_search_docs)
            for chunk in image_search_chain_with_history.stream(
                {"input": final_query, "context": image_context},
                config={"configurable": {"session_id": SESSION_ID}},
            ):
                token = chunk.content if hasattr(chunk, "content") else str(chunk)
                if token:
                    if first_token_time is None:
                        first_token_time = time.time()
                    print(token, end="", flush=True)

        if intent == "outfit":
            outfit_context = build_outfit_context(final_query, gender, user_profile)
            if not outfit_context:
                print("[Layer B không khớp → fallback RAG search]")
                intent = "search"
            else:
                for chunk in outfit_chain_with_history.stream(
                    {"input": user_input, "outfit_context": outfit_context},
                    config={"configurable": {"session_id": SESSION_ID}},
                ):
                    token = chunk.content if hasattr(chunk, "content") else str(chunk)
                    if token:
                        if first_token_time is None:
                            first_token_time = time.time()
                        print(token, end="", flush=True)

        if intent == "search":
            for chunk in full_chat_chain.stream(
                {"input": final_query},
                config={
                    "configurable": {"session_id": SESSION_ID},
                    "callbacks":    [SpyRetrieverHandler()],
                },
            ):
                if "answer" in chunk:
                    if first_token_time is None:
                        first_token_time = time.time()
                    print(chunk["answer"], end="", flush=True)

        end_time = time.time()
        if first_token_time is None:
            first_token_time = end_time

        print(f"\n\nTTFT: {first_token_time - start_time:.2f}s | "
              f"Tổng: {end_time - start_time:.2f}s")

    except Exception as e:
        print(f"\n[LỖI] {e}")

    print("\n\n" + "-" * 60 + "\n")


  CHATBOT TƯ VẤN THỜI TRANG  
  Nhập '0' để thoát | Nhập đường dẫn ảnh nếu có

[Intent: GREETING | Gender: female | Profile: chưa có]
Chatbot: Xin chào! Mình là trợ lý tư vấn thời trang của shop. Bạn cần tìm sản phẩm hay muốn được gợi ý phối đồ hôm nay? 😊

------------------------------------------------------------

[Intent: GREETING | Gender: female | Profile: chưa có]
Chatbot: Xin chào! Mình là trợ lý tư vấn thời trang của shop. Bạn cần tìm sản phẩm hay muốn được gợi ý phối đồ hôm nay? 😊

------------------------------------------------------------

[Intent: SEARCH | Gender: female | Profile: chưa có]
Chatbot: 
[DEBUG] Câu hỏi sau rewrite: tôi muốn tìm 1 bộ đồ vest


[DEBUG] Câu hỏi sau rewrite: tôi muốn tìm 1 bộ đồ vest



Hiện tại, trong danh mục sản phẩm tôi có, **không có sản phẩm nào là bộ đồ vest**.  

Tuy nhiên, nếu bạn quan tâm đến phong cách trang phục nam, tôi có thể gợi ý một số lựa chọn phù hợp khác như:  

1. **Áo tank top nam RaanPahMuang in họa tiết Daddy Longlegs**  
- Mã SP: B0755T1MS3  
- Giá: 716.880 VND  
- Đặc điểm: Áo ba lỗ, họa tiết nghệ thuật, thoáng mát  
- Lý do phù hợp: Thích hợp cho mùa hè, thể thao hoặc đi chơi, mang lại vẻ ngoài cá tính  
- Ảnh: ![Áo tank top]({{IMAGE_URL}})  

2. **Áo ghi lê đa năng nam WINOTO**  
- Mã SP: B08HCS31W4  
- Giá: 718.800 VND  
- Đặc điểm: Áo khoác ghi lê, nhiều túi, tiện dụng  
- Lý do phù hợp: Dành cho hoạt động ngoài trời, du lịch, mang tính đa năng  
- Ảnh: ![Áo ghi lê]({{IMAGE_URL}})  

3. **Khăn choàng cổ vô cực BIAOSU**  
- Mã SP: B07ZR9N9H8  
- Giá: 527.760 VND  
- Đặc điểm: Khăn choàng cổ, chất liệu cotton-polyester, có túi ẩn  
- Lý do phù hợp: Phù hợp cho du lịch hoặc mặc bên ngoài áo, tạo vẻ ấm áp và hiện đại  
- Ảnh: ![Khăn choàng cổ